<a href="https://colab.research.google.com/github/DennisForge/ml-product-reviews-project/blob/main/notebook/product_reviews_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 📚 Loading and inspecting the dataset
Before diving into analysis, we first need to load the dataset and take a look at its structure.
In this step, we will:
- Load the CSV file from GitHub
- Check how many rows and columns we have
- Display the first few rows
- Review data types and basic metadata for each column
This will help us ensure the dataset is correctly loaded and ready for further exploration.

In [ ]:
import pandas as pd

# load dataset from GitHub
url = "https://raw.githubusercontent.com/DennisForge/ml-product-reviews-project/main/data/product_reviews_full.csv"
df = pd.read_csv(url)

# Print shape (number of rows and columns)
print("Dataset shape (rows, columns):", df.shape)

# Show first 5 rows
print("\nFirst 5 rows:")
display(df.head())

# Show column data types and non-null counts
print("\nDataset info:")
df.info()

## 🔎 Checking for missing values

Missing data can cause problems during model training or analysis.
Here, we will:
- Count the number of missing (NaN) values per column
- Visualize missing values using a heatmap

This will help us identify any columns that require cleaning or imputation.

In [ ]:
# Count missing values per column
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# Visualize missing data with seaborn heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.heatmap(df.isna(), cbar=False, cmap="YlOrRd")
plt.title("Missing Values Heatmap")
plt.show()

## 📊 Sentiment analysis
Let's check how many reviews are labeled as positive vs negative vs neutral.
This helps us:
- Understand the balance between classes
- Detect if the dataset is skewed

In [ ]:
# Count occurrences of each sentiment label
sentiment_counts = df['sentiment'].value_counts()

# Print counts
print("Sentiment distribution (counts):")
print(sentiment_counts)

In [ ]:
# Plot sentiment distribution
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.bar(sentiment_counts.index, sentiment_counts.values, color=['skyblue', 'salmon'])
plt.title("Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Number of Reviews")
plt.show()

## 📁 Exploring the `product_price` Column

Before we can use price data in any meaningful way, we need to understand how it is stored and formatted.

In this section, we will:
- Check the data type of the `product_price` column,
- Preview a few sample values,
- Identify the most common price entries,
- Detect non-numeric or problematic values such as `"Free"`, `"N/A"`, or
corrupted symbols.

 ⚒️ This is an important part of data cleaning.
Even if a value *Looks* like a
number (e.g. `$49.19`), it may still be stored as a string and cause problems
during numeric analysis or modeling.
Let's investigate what we're working with!

In [ ]:
# 1. Check the data type of the 'product_price' column
print("Data type of product_price column:", df['product_price'].dtype)

# 2. Display the first 10 values from the column
print("\nFirst 10 values in product_price column:")
print(df['product_price'].head(10))

# 3. Show the 20 most frequent values in the column
print("\nTop 20 most frequent values in product_price column:")
print(df['product_price'].value_counts().head(20))

# 4. Check for known non-numeric text values
problematic_values = ['Free', 'Not Available', 'N/A', 'None', '-', 'free', 'unknown', 'Unavailable']
# Identify rows containing these specific non-numeric values
mask_problematic = df['product_price'].astype(str).str.strip().isin(problematic_values)
df_problematic = df[mask_problematic]
print(f"\nFound {len(df_problematic)} rows with problematic textual values:")
display(df_problematic[['product_price']].drop_duplicates())

# 5. Find and display some of the non-numeric values
price_clean = df['product_price'].astype(str).str.strip()
price_numeric = pd.to_numeric(price_clean, errors='coerce')

invalid_prices = df[price_numeric.isna()]
print("Number of non-numeric prices:", len(invalid_prices))
display(invalid_prices[['product_price']].drop_duplicates().head(20))

## 🧹 Removing missing values

We already analyzed missing data in the previous step.
Now we will simply drop all rows that contain missing values,
and check the new shape and count missing values per column.

In [ ]:
# Drop all rows with missing values
df = df.dropna()

# Display new shape of the dataset
print("New dataset shape:", df.shape)

# Count missing values per column
print("Missing values per column:")
print(df.isna().sum())

## 🔎 Summary of Data Cleaning Steps
1. Total rows before cleaning
Calculate the initial size of the dataset to understand the full scope of available data.
2. Rows remaining after `dropna()`
Count how many rows remain once all entries with missing values are removed.
3. Percentage of data lost
Compute what fraction of the dataset was removed by the cleaning process. This shows how aggressive the data loss is and whether additional strategies (e.g., imputation) may be needed.
4. Bonus: Inspect dropped rows
Randomly sample several rows that would be deleted. Review them to see if they contain partially useful information, unusual patterns, or values worth keeping or imputing instead of discarding.

In [ ]:
import pandas as pd


# load dataset from GitHub
url = "https://raw.githubusercontent.com/vladimir-dresevic/ml-product-reviews-project/main/data/product_reviews_full.csv"


df = pd.read_csv(url)

# Count the number of rows before removing missing values
rows_before = len(df)

# Filter out rows that contain at least one missing value
rows_with_nan = df[df.isnull().any(axis=1)]

# Display a random sample of rows that will be removed
print(" Randomly selected rows containing missing values:\n")
print(rows_with_nan.sample(n=min(5, len(rows_with_nan)), random_state=42))

# Remove rows with any missing values
df_cleaned = df.dropna()

# Count the number of rows after removing missing values
rows_after = len(df_cleaned)

# Show removal statistics
print("\n Removal statistics:")
print(f"- Number of rows before: {rows_before}")
print(f"- Number of rows after: {rows_after}")
print(f"- Number of removed rows: {rows_before - rows_after}")

## 💰 Parsing the `product_price` column

we neticed that some produce prices are stored as numbers,
while others contain a currency prefix like `"$"` (e.g. `"48.18"` or `"$13190"`).

To make this column usable, we will:
- Remove the `"$"` text and any extra characters,
- Convert all values to numbers (`float`),
- Drop invalid rows if conversion fails.




In [ ]:
# Step 1: Convert to string and remove the 'USD' prefix and any leading/trailing spaces
df['product_price_cleaned'] = (
    df['product_price']
    .astype(str)
    .str.replace(r'$', '', regex=True)    # Remove '$'
    .str.replace(r'[^\d.]', '', regex=True) # Remove all non-numeric characters except the dot
    .str.strip()
)
# Step 2: Convert cleaned string to float
df['product_price'] = pd.to_numeric(df['product_price_cleaned'], errors='coerce')

# Step 3: Drop the temporary column
df = df.drop(columns=['product_price_cleaned'])

# Step 4: Drop any rows where conversion failed (still NaN)
df = df.dropna(subset=['product_price'])

# Step 5: Confirm result
print("Column type after parsing:", df['product_price'].dtype)
print("\nPrice summary:")
print(df['product_price'].describe())

## ✅ standardizing the `sentiment` column

The `sentiment` column should only contain `"positive"`, `"negative"` and
`"neutral"` values.

We will:
- Convert all values to lowercase,
- Convert column type to `'category'`
- Check results

In [ ]:
# Step 1: Convert all sentiment values to lowercase and strip extra spaces
df['sentiment'] = df['sentiment'].astype(str).str.lower().str.strip()

# Step 2: Show all unique values in the sentiment column
print("Unique sentiment values after cleaning:")
print(df['sentiment'].value_counts())

# Step 3: Convert column type to 'category'
df['sentiment'] = df['sentiment'].astype('category')
print("\nSentiment column converted to type:", df['sentiment'].dtype)

## 🔪 Removing irrelevant columns

We will now remove columns that are not useful for model training:

- `review_uuid` - just a unique ID,
- `product_name` too specific and inconsistent.

The key features we want to keep are:

- `review_title` - short description of whole review,
- `review_text` - the main input for sentiment prediction,
- `product_price` - to be analyzed further,
- `sentiment` - the target variable.

In [ ]:
# Drop columns that are not useful for modeling
df = df.drop(columns=['review_uuid', 'product_name'])

# Preview remaining columns
print("Remaining columns:")
print(df.columns.tolist())

## 💶 Does product price affect sentiment?

Let's explore whether the price of a product has any influence on the sentiment of reviews.

We will:

- Look at summary statistics of product prices per sentiment,
- Visualize the price distribution grouped by sentiment.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Show summary statistics grouped by sentiment
print("Price summary by sentiment:")
print(df.groupby('sentiment', observed=False)['product_price'].describe())

# Boxplot of prices by sentiment
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sentiment', y='product_price')
plt.title("Product Price Distribution by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Product Price")
plt.grid(True)
plt.show()

## 📊 Creating a new feature - `review_length`

We will now create a new numeric feature called `review_length` that represents the number of characters in each review.

Then we'll visualize how review length varies across different sentiment categories.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Create new column with length of each review_text
df['review_length'] = df['review_text'].astype(str).str.len()

# Show basic stats
print("Review length summary:")
print(df['review_length'].describe())

# Group by sentiment and describe review length
print("Review length statistics by sentiment:")
print(df.groupby('sentiment', observed=False)['review_length'].describe())

# Visualize distribution of review length by sentiment
plt.figure(figsize=(10, 5))
sns.boxplot(data=df, x='sentiment', y='review_length')
plt.title("Review Length by Sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Review Length (number of characters)")
plt.grid(True)
plt.show()

## 📚 Training and Comparing Multiple Machine Learning Models

Train and evaluate several different machine learning models in order to find the best one for our classification task.

We will go through the following steps:

• Split the dataset into training and test sets,
• Prepare the data by:
 - transforming the `review_title` and `review_text` columns using `TF-IDF`,
 - scaling the `review_length` column using `MinMaxScaler`,
• Use a `ColumnTransformer` to combine all features into a single input matrix,
• Define and train `five different classification algorithms`,
• Wrap all components into a unified `Pipeline` for each model,
• Evaluate model performance using `classification reports`,
• Save all results to a text file for later review.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report

import os

# Features and label
X = df[["review_title", "review_text", "review_length"]]
y = df["sentiment"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessor: TF-IDF for text, MinMaxScaler for numeric feature
preprocessor = ColumnTransformer(
    transformers=[
        ("title", TfidfVectorizer(), "review_title"),
        ("text", TfidfVectorizer(), "review_text"),
        ("length", MinMaxScaler(), ["review_length"])
    ]
)

# List of classifiers
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Support Vector Machine": LinearSVC()
}

# Create results directory if it doesn't exist
os.makedirs('../results', exist_ok=True)

# Train and evaluate - capture results
results = []

for name, model in models.items():
    print(f"\n🔎 {name}")
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("classifier", model)
    ])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Capture the report
    report = classification_report(y_test, y_pred)
    print(report)
    results.append(f"\n🔎 {name}\n{report}")

# Save all results to a text file
with open('../results/model_comparison.txt', 'w') as f:
    f.write('\n'.join(results))

print("\n✅ Results saved to results/model_comparison.txt")